# 생성형 AI 기반 요약문 작성

## 요약

요약은 긴 내용을 짧게 줄이되, 대상자에게 필요한 핵심은 남기는 일을 의미한다. 항만 업무를 놓고보았을때 선박 한 척의 적하목록 수백줄을 보고 '냉동 40개, 위험물2개 포함'과 같은 요약을 전하는 것과 같다.

### 요약 방법

요약을 위한 방법은 크게 두가지로 구분해 볼 수 있다.

- 추출 요약: 원문에 이미 있는 문장 중에서 중요한 것들을 골라서 그대로 이어붙이는 방식
- 생성 요약: 원문의 내용을 이해한다음에 새로운 문장으로 다시 작성하는 방식

생성형 AI를 이용해서 요약 -> 생성요약

## 좋은 요약의 기준

요약의 품질 개선을 위해서는 어떠한 요약이 좋은 요약인지에 대한 기준이 팔요하다.

- 정확성: 숫자와 방향이 사실과 같은가
- 근거성: 모든 문장이 사실 묶음에 근거하는가
- 포괄성: 중요한 사실을 누락하지 않았는가
- 간결성: 충분히 짧은가


In [4]:
def fact_sentence(f) :
    value = f"{f['value']:,}" if isinstance(f["value"], int) else f"{f['value']}"
    unit = f["unit"] if f["unit"] == "%" else f" {f['unit']}"
    return f"{f['label']}은/는 {value}{unit}입니다 [{f['id']}]."

example_facts = [
    {"id": "A1", "label": "이번 달 물동량", "value": 2_100_000, "unit": "TEU"},
    {"id": "A2", "label": "전년 동월 대비 증감률", "value": 3.2, "unit": "%"},
    {"id": "A3", "label": "수출 증감률", "value": 4.1, "unit": "%"},
    {"id": "A4", "label": "수입 증감률", "value": -1.8, "unit": "%"},
    {"id": "A5", "label": "환적 비중", "value": 56.0, "unit": "%"},
]  # 예시 수치

sentences = [fact_sentence(f) for f in example_facts]

sentences

['이번 달 물동량은/는 2,100,000 TEU입니다 [A1].',
 '전년 동월 대비 증감률은/는 3.2%입니다 [A2].',
 '수출 증감률은/는 4.1%입니다 [A3].',
 '수입 증감률은/는 -1.8%입니다 [A4].',
 '환적 비중은/는 56.0%입니다 [A5].']

In [3]:
import re

def split_sentence(text):
    return [ s.strip() for s in re.split(r'(?<=[.])\s+', text) if text.strip() ]

def summary_metrics(summary_text, source_text, idx):
    """작성된 요약문에 대한 지표를 반환하는 함수

    Args:
        summary_text: 생성된 요약문 문자열
        source_text: : 원본 보고서 문자열
        idx: 지표 항목

    Returns:
        지표 딕셔너리
    """
    sents = split_sentence(summary_text)
    sections = set(re.findall(r'\[(A-Z)\d{1,2}]', summary_text))

    return {
        '근거성': sum(bool(re.search(r'\[(A-Z)\d{1,2}]', s)) for s in sents) / len(sents),
        '포괄성': len(sections & set(idx)) / len(idx),
        '입축률': len(summary_text) / len(source_text),
    }


In [8]:
source_text = " ".join(sentences)

important = [ 'A1', 'A2', 'A5' ]

ext = " ".join([sentences[0], sentences[1]])
suggestion = '이번 달 물동량은 약 210만 TEU로서 전년 같은 달보다 3.2% 늘었고, 수출은 늘고 수입은 줄었습니다. [A1][A2][A3][A4].'

print(ext)
print(suggestion)

contents = {
    '추출 2문장',
    '생성 1문장',
    '근거 없는 문장 섞임'
}


이번 달 물동량은/는 2,100,000 TEU입니다 [A1]. 전년 동월 대비 증감률은/는 3.2%입니다 [A2].
이번 달 물동량은 약 210만 TEU로서 전년 같은 달보다 3.2% 늘었고, 수출은 늘고 수입은 줄었습니다. [A1][A2][A3][A4].
